# WarehousePG Backups and Disaster Recovery
WarehousePG Disaster Recovery (whpg-dr) protects WarehousePG (WHPG) clusters against data loss and extended outages. It takes consistent physical backups of the entire cluster, ships WAL to a secondary storage repository, so you can restore the cluster to any named restore point, on the original hosts or on different infrastructure.

*Note: This notebook is not compatible with `warehousepg_s3.yaml` template at this time.*

## Init
* Set variable names
* Remove any leftover backups from previous demos
* Remove any existing configuration files
* Connect to WarehousePG

In [1]:
# Variables for demo
cluster_name = "dr_demo_cluster"
bucket_name = "whpg-backups-us-east-2"
prefix = "demo-dr-backups"
region = "us-east-2"
folder = "whpg-demo"

!pip install psycopg

# Cleanup any existing backups 
!aws s3 rm --recursive s3://{bucket_name}/{prefix}
!rm -rf /home/gpadmin/.whpg-dr

# Connect to the database
from sqlalchemy import create_engine
PGUSER="gpadmin"
PGHOST="cdw"
PGPORT="5432"
PGDATABASE="dev"
conn = create_engine(f"postgresql://{PGUSER}@{PGHOST}:{PGPORT}/{PGDATABASE}")

# If running elsewhere without trust auth, set a password and use this instead:
# PGPASSWORD="your_password_here"
# conn = create_engine(f"postgresql://{PGUSER}:{PGPASSWORD}@{PGHOST}:{PGPORT}/{PGDATABASE}")

%reload_ext sql
%sql conn

delete: s3://whpg-backups-us-east-2/demo-dr-backups/dr_demo_cluster/basebackup/dr_demo_cluster-seg-1/base/20260925T115521/backup.info
delete: s3://whpg-backups-us-east-2/demo-dr-backups/dr_demo_cluster/basebackup/dr_demo_cluster-seg0/base/20260925T115521/data.tar
delete: s3://whpg-backups-us-east-2/demo-dr-backups/dr_demo_cluster/restore-points/20260925T115520_base_backup/20260925-115531R_whpgdr_full_backup
delete: s3://whpg-backups-us-east-2/demo-dr-backups/dr_demo_cluster/cluster_info.yaml
delete: s3://whpg-backups-us-east-2/demo-dr-backups/dr_demo_cluster/restore-points/20260925T115520_base_backup/20260925-115548R
delete: s3://whpg-backups-us-east-2/demo-dr-backups/dr_demo_cluster/wal/dr_demo_cluster-seg-1/wals/0000000100000000/000000010000000000000004.gz
delete: s3://whpg-backups-us-east-2/demo-dr-backups/dr_demo_cluster/basebackup/dr_demo_cluster-seg0/base/20260925T115521/backup.info
delete: s3://whpg-backups-us-east-2/demo-dr-backups/dr_demo_cluster/basebackup/dr_demo_cluster-seg

## Metadata
Capture metadata about the cluster to files for the demo.

In [2]:
result = %sql SELECT DISTINCT hostname FROM gp_segment_configuration WHERE role = 'p';
with open("all_nodes.txt", "w") as f:
    for row in result:
        f.write(row.hostname + "\n")
print(open("all_nodes.txt").read())

Running query in 'postgresql://gpadmin@cdw:5432/dev'

1 rows affected.

cdw



In [3]:
result = %sql SELECT DISTINCT hostname FROM gp_segment_configuration WHERE role = 'p' AND content >= 0;
with open("segment_nodes.txt", "w") as f:
    for row in result:
        f.write(row.hostname + "\n")

print(open("segment_nodes.txt").read())

Running query in 'postgresql://gpadmin@cdw:5432/dev'

1 rows affected.

cdw



In [4]:
result = %sql SELECT DISTINCT'/' || split_part(datadir, '/', 2) || '/' || split_part(datadir, '/', 3) AS datadir FROM gp_segment_configuration WHERE role = 'p' AND content > 0;
with open("data_directories.txt", "w") as f:
    for row in result:
        f.write(row.datadir + "\n")

print(open("data_directories.txt").read())

Running query in 'postgresql://gpadmin@cdw:5432/dev'

1 rows affected.

/data1/primary



## Create Table

This table will be used to verify that the backup was taken and restored.

In [5]:
%%sql
DROP SCHEMA IF EXISTS foo CASCADE;
CREATE SCHEMA foo;

CREATE TABLE foo.bar AS SELECT i FROM generate_series(1, 100000) as i DISTRIBUTED BY (i);

SELECT *
FROM foo.bar
LIMIT 10;

Running query in 'postgresql://gpadmin@cdw:5432/dev'

100000 rows affected.

10 rows affected.

i
1
5
11
12
14
15
17
20
23
25


In [6]:
# Close the database connection
connection_url = f"postgresql://{PGUSER}@{PGHOST}:{PGPORT}/{PGDATABASE}"
%sql --close {{connection_url}}

## Disaster Recovery Utility
The `whpg-dr` commands are typically executed from the command line on the coordinator node but are being executed here in a Notebook for the demo.

### Create yaml configuration file
This file is used to configure backups to S3.

In [7]:
config = f"""
cluster_name: {cluster_name}
storage:
  type: s3
  bucket: {bucket_name}
  prefix: {prefix}
  region: {region}

barman_options:
  compression: gzip
  compression_level: 6
"""

with open("whpg-dr_demo.yaml", "w") as f:
    f.write(config)

print(open("whpg-dr_demo.yaml").read())
    


cluster_name: dr_demo_cluster
storage:
  type: s3
  bucket: whpg-backups-us-east-2
  prefix: demo-dr-backups
  region: us-east-2

barman_options:
  compression: gzip
  compression_level: 6



## Configure Backup
Store the metadata needed for disaster recovery backups and enable WAL archiving.

In [8]:
!echo y | whpg-dr configure backup /home/gpadmin/whpg-dr_demo.yaml

Generating barman configs...
This command will generate barman configuration files:
  /home/gpadmin/.whpg-dr/dr_demo_cluster/backup/config.yaml
  /home/gpadmin/.whpg-dr/dr_demo_cluster/backup/barman_conf/dr_demo_cluster-global.conf
  /home/gpadmin/.whpg-dr/dr_demo_cluster/backup/barman_conf/dr_demo_cluster-seg-1.conf
  /home/gpadmin/.whpg-dr/dr_demo_cluster/backup/barman_conf/dr_demo_cluster-seg0.conf
  /home/gpadmin/.whpg-dr/dr_demo_cluster/backup/barman_conf/dr_demo_cluster-seg1.conf
It will run gpconfig and restart the cluster (gpstop -ar) to apply archive settings.

Proceed? (y/N): Distributing barman configs to cluster hosts (/home/gpadmin/.whpg-dr/dr_demo_cluster/backup)...
Segment topology snapshot saved: /home/gpadmin/.whpg-dr/dr_demo_cluster/backup/segment_configuration.csv
Barman configuration generated.
Restarting the cluster (gpstop -ar) to apply archive GUCs. Please wait...
Barman archive GUCs configured (archive_mode, archive_command).
Suggestion: run `whpg-dr check dr_de

### Check 
Make sure the configure backup command worked properly.

In [9]:
!whpg-dr check {cluster_name}

Running barman check for server dr_demo_cluster-seg-1
Running barman check for server dr_demo_cluster-seg0
Running barman check for server dr_demo_cluster-seg1
Server dr_demo_cluster-seg0:
	PostgreSQL: OK
	superuser or standard user with backup privileges: OK
	wal_level: OK
	directories: OK
	retention policy settings: OK
	backup maximum age: OK (no last_backup_maximum_age provided)
	backup minimum size: OK (0 B)
	cloud storage configuration: OK
	wal maximum age: OK (no last_wal_maximum_age provided)
	wal size: OK (0 B)
	compression settings: OK
	failed backups: OK (there are 0 failed backups)
	minimum redundancy requirements: OK (have 0 non-incremental backups, expected at least 0)
	systemid coherence: OK (no system Id stored on disk)
	archive_mode: OK
	archive_command: OK
	continuous archiving: OK
	archiver errors: OK
Server dr_demo_cluster-seg1:
	PostgreSQL: OK
	superuser or standard user with backup privileges: OK
	wal_level: OK
	directories: OK
	retention policy settings: OK
	backu

## Take a backup

In [10]:
!whpg-dr backup {cluster_name}


Step 1/3: Pre-backup checks...
Running barman checks before backup...
Barman checks passed

Step 2/3: Running barman backup...
Starting backup for 1 coordinator and 2 segments with cluster: dr_demo_cluster (running on segment hosts)
Running barman backup for server: dr_demo_cluster-seg-1 (content -1)
Running barman backup for server: dr_demo_cluster-seg1 (content 1)
Running barman backup for server: dr_demo_cluster-seg0 (content 0)
All backups completed successfully (1 coordinator and 2 segments)

Step 3/3: Creating restore point...
Per-backup segment configuration saved: s3://whpg-backups-us-east-2/demo-dr-backups/dr_demo_cluster/restore-points/20260925T121949_base_backup/segment_configuration.csv
Waiting for WAL archiving to complete for cluster dr_demo_cluster...
Restore point created: 20260925-122000R_whpgdr_full_backup (19s)
Restore point metadata: s3://whpg-backups-us-east-2/demo-dr-backups/dr_demo_cluster/restore-points/20260925T121949_base_backup/20260925-122000R_whpgdr_full_b

## List backups

In [11]:
!whpg-dr list-backup {cluster_name}

Listing backups...
Listing restore points...
Full backup: 20260925T121949_base_backup
└── Restore Points:
    └── 20260925-122000R_whpgdr_full_backup: 2026-09-25 12:20:00



## Restore Point
At this point, an initial backup has been created and WAL archive is copied to S3. You can recover this cluster now.

A restore point provides point-time-recovery of the cluster. Changes to your cluster are tracked in WAL files and using the `create-restore-point` command provides a way to take a backup of just the changes (incremental) since the last full backup was taken.

Note: You don't need to create a restore point immediately after taking a backup.

In [12]:
!whpg-dr create-restore-point {cluster_name}

Waiting for WAL archiving to complete for cluster dr_demo_cluster...
Restore point created: 20260925-122017R (5.1s)
Restore point metadata: s3://whpg-backups-us-east-2/demo-dr-backups/dr_demo_cluster/restore-points/20260925T121949_base_backup/20260925-122017R


## Additional Restore Point
After executing `create-restore-point`, you see that there is an additional point in time to restore the backup to.

You should see two Restore Points. The first was created with the full backup and the second was just taken moments later.

In [13]:
!whpg-dr list-backup {cluster_name}

Listing backups...
Listing restore points...
Full backup: 20260925T121949_base_backup
└── Restore Points:
    ├── 20260925-122000R_whpgdr_full_backup: 2026-09-25 12:20:00
    └── 20260925-122017R: 2026-09-25 12:20:17



## Restore
A restore is typically performed on a different cluster but for this demo, we will use the same one. 
### Steps
* Create the restore configuration file
* Shutdown WarehousePG
* Delete all database files
* Restore latest Restore Point
* Promote the cluster
* Validate

In [14]:
import os

coordinator = "cdw"

with open("data_directories.txt") as d:
    data_directories = "\n".join(f"  - {line.strip()}" for line in d if line.strip())

with open("segment_nodes.txt") as f:
    segment_hosts = "\n".join(f"  - {line.strip()}" for line in f if line.strip())
    
if os.path.isdir("/s3data"):
    coordinator_data_dir = "/data/coordinator"
else:
    coordinator_data_dir = "/data1/coordinator"

config = f"""source_cluster_name: {cluster_name}

storage:
  type: s3
  bucket: {bucket_name}
  prefix: {prefix}
  region: {region}
  credential_source: default

coordinator_host: {coordinator}
coordinator_data_directory: {coordinator_data_dir}

segment_hosts:
{segment_hosts}

data_directory: 
{data_directories}

data_directory_prefix: gpseg
"""

with open("restore.yaml", "w") as f:
    f.write(config)

print(open("restore.yaml").read())

source_cluster_name: dr_demo_cluster

storage:
  type: s3
  bucket: whpg-backups-us-east-2
  prefix: demo-dr-backups
  region: us-east-2
  credential_source: default

coordinator_host: cdw
coordinator_data_directory: /data1/coordinator

segment_hosts:
  - cdw

data_directory: 
  - /data1/primary

data_directory_prefix: gpseg



## Remove the existing WarehousePG cluster
This step is only needed when you are restoring to an existing cluster. 

In [15]:
import glob
import shutil
import os

!sudo systemctl stop wem
!sudo systemctl stop clickhouse-server

!pxf cluster stop || true 
!gpstop -M immediate -a || true
!gpssh -f all_nodes.txt -e "rm -rf /data1/coordinator/* /data[1-8]/primary/* /data[1-8]/mirror*"

# Checks for /s3data used for the S3Files + EFS template
if os.path.isdir("/s3data"):
    for pattern in ["/s3data/whpg/*", "/data/coordinator/*", "/data/primary/*"]:
        for path in glob.glob(pattern):
            if os.path.isdir(path) and not os.path.islink(path):
                shutil.rmtree(path)
            else:
                os.remove(path)

Stopping PXF on coordinator host and 0 segment hosts...
PXF stopped successfully on 1 out of 1 host
20260925:12:20:49:085253 gpstop:cdw:gpadmin-[INFO]:-Starting gpstop with args: -M immediate -a
20260925:12:20:49:085253 gpstop:cdw:gpadmin-[INFO]:-Gathering information and validating the environment...
20260925:12:20:49:085253 gpstop:cdw:gpadmin-[INFO]:-Obtaining Greenplum Coordinator catalog information
20260925:12:20:49:085253 gpstop:cdw:gpadmin-[INFO]:-Obtaining Segment details from coordinator...
20260925:12:20:49:085253 gpstop:cdw:gpadmin-[INFO]:-Greenplum Version: 'postgres (Greenplum Database) 7.6.0-WHPG build commit:0f73ef6e0729cbf71cf225b41d58e1bd1090f335'
20260925:12:20:49:085253 gpstop:cdw:gpadmin-[INFO]:-Commencing Coordinator instance shutdown with mode='immediate'
20260925:12:20:49:085253 gpstop:cdw:gpadmin-[INFO]:-Coordinator segment instance directory=/data1/coordinator/gpseg-1
20260925:12:20:49:085253 gpstop:cdw:gpadmin-[INFO]:-Attempting forceful termination of any lef

## Configure Restore

In [16]:
!echo y | whpg-dr configure restore /home/gpadmin/restore.yaml

Using backup from latest restore point: 20260925T121949_base_backup
Reading segment configuration from S3: s3://whpg-backups-us-east-2/demo-dr-backups/dr_demo_cluster/restore-points/20260925T121949_base_backup/segment_configuration.csv
Read segment config from s3://whpg-backups-us-east-2/demo-dr-backups/dr_demo_cluster/restore-points/20260925T121949_base_backup/segment_configuration.csv: content IDs [-1 0 1]
This command will generate barman configuration files:
  /home/gpadmin/.whpg-dr/dr_demo_cluster/restore/config.yaml
  /home/gpadmin/.whpg-dr/dr_demo_cluster/restore/restore.json
  /home/gpadmin/.whpg-dr/dr_demo_cluster/restore/barman_conf/dr_demo_cluster-global.conf
  /home/gpadmin/.whpg-dr/dr_demo_cluster/restore/barman_conf/dr_demo_cluster-seg-1.conf
  /home/gpadmin/.whpg-dr/dr_demo_cluster/restore/barman_conf/dr_demo_cluster-seg0.conf
  /home/gpadmin/.whpg-dr/dr_demo_cluster/restore/barman_conf/dr_demo_cluster-seg1.conf
It will distribute configs to segment hosts.

Proceed? (y/N

## Restore Latest

In [17]:
!echo y | whpg-dr restore {cluster_name} --target-name latest


Step 1/4: Resolving and verifying restore target...
Resolved latest restore point: 20260925-122017R (backup: 20260925T121949_base_backup)
Verifying topology compatibility between backup 20260925T121949_base_backup and restore configuration...
Checking if coordinator port 5432 is available on cdw...
Verifying backup 20260925T121949_base_backup integrity...
Finding backup...
Checked backup 20260925T121950 on server dr_demo_cluster-seg0
Skipping inactive server 'dr_demo_cluster-seg0'
Checked backup 20260925T121950 on server dr_demo_cluster-seg-1
Skipping inactive server 'dr_demo_cluster-seg-1'
Checked backup 20260925T121950 on server dr_demo_cluster-seg1
Skipping inactive server 'dr_demo_cluster-seg1'
All backup checks completed successfully

Step 2/4: Restoring base backup...
Restoring cluster dr_demo_cluster from backup 20260925T121949_base_backup
Using global config: /home/gpadmin/.whpg-dr/dr_demo_cluster/restore/barman_conf/dr_demo_cluster-global.conf
Found 1 coordinator and 2 segmen

## Cluster Status
The cluster is now available for additional WAL archives to be applied. In a disaster recovery configuration, the second cluster located in a different region would be ready for more Recovery Points to be applied or be promoted to a active cluster.

## Promote the Cluster
Make the cluster available.

In [18]:
!whpg-dr promote {cluster_name}
!gpstart -a
!pxf cluster start
!sudo systemctl start clickhouse-server
!sudo systemctl start wem




Step 1/6: Running pre-promote checks...
Checking segment consistency (pg_controldata)...
All segments are in "shut down in recovery" state.
Promoting cluster dr_demo_cluster
Coordinator: host=cdw, port=5432, datadir=/data1/coordinator/gpseg-1
Total to update: 1 coordinator and 2 segments

Step 2/6: Configuring ports and recovery settings on all segments...
All segments configured.

Step 3/6: Starting coordinator in utility mode (pg_ctl)...
Coordinator started successfully.

Step 4/6: Updating gp_segment_configuration...
  Updating content -1: host=cdw, port=5432, datadir=/data1/coordinator/gpseg-1
  Updating content 0: host=cdw, port=6000, datadir=/data1/primary/gpseg0
  Updating content 1: host=cdw, port=6001, datadir=/data1/primary/gpseg1
Segment configuration updated successfully.

Step 5/6: Disabling synchronous replication settings...
Synchronous replication disabled (synchronous_standby_names='') in postgresql.auto.conf.

Step 6/6: Stopping coordinator (pg_ctl)...
Coordinator st

## Reconnect to the Database

In [19]:
# Connect to the database
from sqlalchemy import create_engine
PGUSER="gpadmin"
PGHOST="cdw"
PGPORT="5432"
PGDATABASE="dev"
conn = create_engine(f"postgresql://{PGUSER}@{PGHOST}:{PGPORT}/{PGDATABASE}")

# If running elsewhere without trust auth, set a password and use this instead:
# PGPASSWORD="your_password_here"
# conn = create_engine(f"postgresql://{PGUSER}:{PGPASSWORD}@{PGHOST}:{PGPORT}/{PGDATABASE}")

%reload_ext sql
%sql conn

## Verify Cluster was Restored

In [20]:
%%sql
SELECT *
FROM foo.bar
LIMIT 10;

Running query in 'postgresql://gpadmin@cdw:5432/dev'

10 rows affected.

i
2
3
4
6
7
8
9
10
13
16


## Close the Connection

In [21]:
connection_url = f"postgresql://{PGUSER}@{PGHOST}:{PGPORT}/{PGDATABASE}"
%sql --close {{connection_url}}